# 07 — Under the Hood: How CGE-Core Actually Solves a Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/07_under_the_hood.ipynb)

We inspect the Pyomo model, parameters, variables, constraints, numeraire, degrees of freedom, Walras' law, calibrated parameters, and the comparison layer.

We use Hosoe's `stdcge`: rich enough to be interesting, small enough to inspect.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORKSPACE = Path("/content") if Path("/content").exists() else Path.home() / ".cache"
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKSPACE / "CGE-core-colab"

if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main", "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/miraflor/CGE-core.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core
print("✓ CGE-Core", cge_core.__version__)
print("✓ Repository:", REPO_DIR)


subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "amplpy.modules", "install", "coin"],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

module_path = subprocess.check_output(
    [sys.executable, "-m", "amplpy.modules", "path"],
    text=True,
).strip()
os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found after installing the COIN module."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)

## 2. Build the model — but do not solve it yet

In [ ]:
import pandas as pd
from IPython.display import display
from pyomo.environ import Constraint, Param, Var, value

from cge_core import PyCGE, example_data
from cge_core.examples.stdcge_model_def import StdModelDef

cge = PyCGE(StdModelDef())
cge.model_data(example_data("stdcge"))
cge.model_instance("pf", "LAB")

print("Numeraire:", cge.numeraire)
print("LAB factor price fixed?", cge.base.pf["LAB"].fixed)
print(
    "Degrees of freedom BEFORE dropping a redundant equation:",
    cge.degrees_of_freedom(cge.base),
)

### Why is the DOF not zero yet?

Once one price is fixed as the numeraire, Walras' law makes one market-clearing equation redundant. CGE-Core explicitly deactivates one eligible market-clearing equation.

## 3. Drop the redundant labor-market equation

In [ ]:
print("eqpf['LAB'] active before:", cge.base.eqpf["LAB"].active)

cge.model_drop_redundant("eqpf", "LAB")

print("eqpf['LAB'] active after: ", cge.base.eqpf["LAB"].active)
print("Degrees of freedom after drop:", cge.degrees_of_freedom(cge.base))

## 4. Count the Pyomo machinery

In [ ]:
def component_count(model, ctype):
    rows = []
    for component in model.component_objects(ctype, active=None):
        try:
            n = len(list(component.values()))
        except TypeError:
            n = 1
        rows.append({
            "component": component.local_name,
            "entries": n,
            "active": getattr(component, "active", None),
        })
    return pd.DataFrame(rows)

print("Parameters")
display(component_count(cge.base, Param))

print("Variables")
display(component_count(cge.base, Var))

print("Constraints")
display(component_count(cge.base, Constraint))

## 5. Look at actual equation expressions

In [ ]:
candidate_constraints = [
    "eqF",
    "eqM",
    "eqE",
    "eqQ",
    "eqZ",
    "eqpf",
]

for name in candidate_constraints:
    if not hasattr(cge.base, name):
        continue
    component = getattr(cge.base, name)
    entries = list(component.items())
    if not entries:
        continue
    index, item = entries[0]
    print(f"\n{name}[{index}]")
    print(item.expr)

These are equations in a **simultaneous nonlinear system**. IPOPT searches for values of the endogenous variables that satisfy the active system together.

## 6. Solve the calibrated baseline

In [ ]:
cge.model_calibrate(SOLVER)

print("Termination accepted.")
print("Degrees of freedom:", cge.degrees_of_freedom(cge.base))
print("Objective:", value(cge.base.obj))

## 7. Inspect calibrated parameters

In [ ]:
selected_names = [
    "alpha", "beta", "tauz", "taum",
    "deltam", "deltad", "gamma",
    "xie", "xid", "theta",
]

rows = []
for name in selected_names:
    if not hasattr(cge.base, name):
        continue
    component = getattr(cge.base, name)
    entries = component.items() if component.is_indexed() else [(None, component)]
    for index, item in entries:
        rows.append({
            "parameter": name,
            "index": str(index) if index is not None else "",
            "value": value(item),
        })

display(pd.DataFrame(rows))

Calibration recovers parameter values that make the model reproduce the benchmark SAM. Those parameters remain part of the behavioral structure in the counterfactual.

## 8. Apply one shock and inspect the reporting layer

In [ ]:
cge.model_sim()
cge.model_modify_sim("taum", "BRD", 0.0)
cge.model_solve(SOLVER)

comparison = cge.model_compare()

print("Columns returned by model_compare():")
print(list(comparison.columns))

print("\nVariable components in the comparison:")
print(sorted(comparison["component"].unique()))

display(comparison.head(25))

## The full computational picture

```text
ModelDef
   │
   ├─ sets, parameters, variables, equations, objective
   ↓
PyCGE
   ├─ load data
   ├─ instantiate Pyomo model
   ├─ fix closure anchor / numeraire
   ├─ drop one allowed redundant market equation
   ├─ calibrate baseline
   ├─ clone baseline into a scenario
   ├─ edit allowed exogenous components
   ├─ call nonlinear solver
   └─ return structured comparisons
```

The engine is workflow infrastructure. Economic content lives in the model definition and data; the policy question lives in the scenario and closure.

## You have completed the learning path

- **Control Room:** visually explore models and shocks.
- **Documentation:** read exact equations, APIs, validation, and provenance.
- **Source code:** modify or add model definitions and research applications.

**CGE-Core:** https://github.com/miraflor/CGE-core  
**Documentation:** https://miraflor.github.io/CGE-core/  
**Control Room:** https://miraflor.github.io/CGE-core/control-room/